# 实验一 · Hello World：Fork-Join 与非确定性

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐ 入门　|　**预计时长**：10–20 分钟

> **实验说明**
> 1. 本实验建立 Pthreads 编程的最小完整框架：线程句柄的管理、线程的派生（Fork）与汇合（Join）、以及线程入口函数的固定签名。
> 2. 本实验采用**两版递进**的方式：先给出**基础版**，运行并观察其行为；随后提出「线程创建中途失败」的问题，引出处理该问题的**健壮版**。两版对照阅读，理解并发程序中「资源清理」这一贯穿全章的主题。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本章绝大多数现象都依赖真正的多核并行。请尽量在**华为鲲鹏多核处理器**上运行；单核环境下，非确定性、数据竞争等现象可能无法完整观察。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明**进程**与**线程**在内存布局上的差异，指出线程之间共享哪些区域、私有哪些区域
- 正确使用 `pthread_create` 与 `pthread_join` 实现 **Fork-Join** 并行
- 说明线程入口函数为何必须采用 `void *f(void *)` 这一固定签名
- 通过实际运行观察调度的**非确定性**，并解释其来源
- 严格区分**进程终止**与**线程终止**，明确「僵尸」概念的成立前提
- 识别线程创建中途失败时的资源清理问题，并给出健壮的处理方式

## 🗺️ 学习路径

1. **准备阶段**：理解进程与线程的区别、Fork-Join 模型、以及三个核心 API 的语义
2. **基础版**：实现最小的多线程程序，派生 N 个线程各自打印一行
   → 观察调度的非确定性，理解「执行顺序不可依赖」这一根本事实
3. **提出问题**：若 `pthread_create` 中途失败，已创建的线程会如何？
   → 辨析进程终止、线程终止与「僵尸」的严格含义
4. **健壮版**：引入对部分失败的对称清理，得到工程上正确的实现
   → 建立「任何退出路径都要清理已获取资源」的意识
5. **反面实验与总结**：观察缺少 `pthread_join` 的后果，归纳全章通用规则

## 1. 背景与动机

第三章中，ARM NEON 通过一条指令同时处理多个数据（SIMD），挖掘的是**单个核心内部**的数据级并行。但单核算力有上限，要继续提升性能，必须让**多个核心同时工作**，这就是线程级并行（Thread-Level Parallelism, TLP）。

在共享内存系统上有两种基本的并行执行体：

<!--
| | 进程（Process） | 线程（Thread） |
|---|---|---|
| 地址空间 | 各自独立 | **同一进程内的线程共享地址空间** |
| 共享内容 | 不共享（需经 IPC 显式传递） | 全局区、堆、代码段、文件描述符表 |
| 私有内容 | 全部 | **栈、寄存器上下文、`errno`、线程 ID、信号掩码** |
| 创建开销 | 大（复制页表等内核结构） | 小（共用大部分内核结构） |
| 通信成本 | 高（需拷贝或映射） | **趋近于零（直接读写同一地址）** |
| 故障隔离 | 强（一个进程崩溃不影响其他） | 弱（一个线程越界写坏内存，整个进程崩溃） |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">进程（Process）</th>
      <th style="text-align: left;">线程（Thread）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">地址空间</td>
      <td style="text-align: left;">各自独立</td>
      <td style="text-align: left;"><strong>同一进程内的线程共享地址空间</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">共享内容</td>
      <td style="text-align: left;">不共享（需经 IPC 显式传递）</td>
      <td style="text-align: left;">全局区、堆、代码段、文件描述符表</td>
    </tr>
    <tr>
      <td style="text-align: left;">私有内容</td>
      <td style="text-align: left;">全部</td>
      <td style="text-align: left;"><strong>栈、寄存器上下文、<code>errno</code>、线程 ID、信号掩码</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">创建开销</td>
      <td style="text-align: left;">大（复制页表等内核结构）</td>
      <td style="text-align: left;">小（共用大部分内核结构）</td>
    </tr>
    <tr>
      <td style="text-align: left;">通信成本</td>
      <td style="text-align: left;">高（需拷贝或映射）</td>
      <td style="text-align: left;"><strong>趋近于零（直接读写同一地址）</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">故障隔离</td>
      <td style="text-align: left;">强（一个进程崩溃不影响其他）</td>
      <td style="text-align: left;">弱（一个线程越界写坏内存，整个进程崩溃）</td>
    </tr>
  </tbody>
</table>

概括而言，线程的内存特征是**大共享，小私有**。

这一特征具有两面性：
- **正面**：线程之间传递数据不需要任何拷贝，一个线程写入全局变量后，另一个线程即可读取到更新后的值。
- **负面**：正因为可以随意读写同一块内存，当多个线程同时读写同一变量时，就会产生**数据竞争**。本章从模块二开始的全部内容，都是在处理这一负面后果。

### 💡 为何选择 Pthreads

POSIX Threads（Pthreads）是 IEEE POSIX 1003.1c 标准定义的线程接口，为类 Unix 系统上的事实标准。与 OpenMP 的编译指令相比，Pthreads 需要手工管理每一个线程，代码相对复杂，但**没有任何隐藏行为**：线程何时创建、何时汇合、临界区在何处，全部由程序员显式处理。

这正是将其置于 OpenMP 之前讲授的原因。Pthreads 之于并发编程，其地位类似于汇编语言之于程序设计：今后使用的 OpenMP、C++11 `std::thread`、线程池等等，其内部都是本章这些原语的封装。

## 2. Fork-Join 模型

Pthreads 采用 Fork-Join（派生—汇合）模型组织并行：

```
主线程  ──┬─────────────────────────────────┬──►  继续执行
          │  fork                     join  │
          ├──► 线程 0 ─────────────────────►┤
          ├──► 线程 1 ─────────────────────►┤
          └──► 线程 2 ─────────────────────►┘
```

1. **Fork**：主线程调用 `pthread_create` 派生若干工作线程。该调用返回后，新线程随即与主线程**并发执行**。
2. **并行段**：各线程独立运行，彼此之间没有任何顺序保证。
3. **Join**：主线程调用 `pthread_join` 逐个等待工作线程结束。所有线程汇合后，并行段才算真正完成。

`pthread_join` 有两个作用，缺一不可：
- **等待（同步）**：阻塞主线程，直至目标线程终止，从而保证并行段的结果在主线程继续之前已全部就绪。
- **回收（资源释放）**：读取目标线程的终止状态，并释放其内核数据结构与栈。可汇合（joinable）的线程终止后，若一直无线程对其调用 `pthread_join`，只要**进程仍然存活**，其占用的资源就不会被释放，从而造成资源泄漏（关于这一表述的精确前提，见第 6 节）。

## 3. 核心 API 与技巧

```c
int pthread_create(pthread_t *thread,              // 输出：新线程的句柄
                   const pthread_attr_t *attr,     // 属性，传 NULL 用默认值
                   void *(*start_routine)(void *), // 入口函数
                   void *arg);                     // 传给入口函数的唯一参数

int pthread_join(pthread_t thread,   // 注意：此处传句柄的值
                 void **retval);     // 接收返回值，不关心则传 NULL
```

- **`pthread_create`**：第一个参数传句柄的**地址**（`&handles[t]`）；成功返回 0，失败返回非 0 错误码。
- **`pthread_join`**：第一个参数传句柄的**值**（`handles[t]`）。
- **入口函数签名**：必须严格为 `void *f(void *)`。写成 `void Hello(long rank)` 无法通过编译。
- **返回值约定**：`pthread_*` 系列函数**直接返回错误码，不设置 `errno`**（这与多数 POSIX 函数不同，也与 `sem_*` 系列不同）。要打印可读信息需用 `strerror(rc)`，而非 `perror`。
- **参数传递技巧**：本实验将循环变量按值转换为 `void *` 传入，线程内再转回 `long`。此处依赖一项**实现定义**（implementation-defined）的假设：在 LP64 数据模型下（64 位 Linux、含鲲鹏所属的 AArch64 均采用此模型），指针与 `long` 等宽，故二者之间的往返转换不损失信息。可移植代码应使用 `intptr_t`。

## 4. 环境准备

下面的单元格检查编译器与硬件环境，并定义本实验统一使用的编译、运行、绘图工具函数。

**请务必留意 CPU 核心数**：本章绝大多数现象都依赖真正的多核并行。核心数为 1 时，线程只能被操作系统分时轮转，非确定性、数据竞争等现象都可能观察不到。

In [ ]:
import platform, subprocess, shutil, sys, os

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：非确定性、数据竞争等现象可能无法观察，")
    print("    建议在华为鲲鹏多核处理器上运行本实验。")
else:
    print("\n✅ 环境就绪：编译器可用，多核可用，可以开始实验！")


### 编译与运行工具函数

全章统一的编译选项为：

```bash
gcc -O3 -fPIC -pthread <source>.c -o <target> -lpthread -lm
```

- `-O3`：开启完整优化。教学代码须在与生产环境一致的优化级别下测量，否则所得加速比没有参考价值。
- `-pthread` / `-lpthread`：链接 POSIX 线程库。`-pthread` 同时作用于预处理与链接，是标准写法。
- **不使用 `-march=native`**：该选项会针对当前机器生成专用指令，使各版本对比失去公平性，且生成的二进制无法在其他机型上运行。

In [ ]:
SRC_DIR = "src_hello"  # 源代码目录
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args, echo=True):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args], capture_output=True, text=True
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


## 5. 基础版实现

第一个版本是最小的多线程程序：派生 `N` 个线程，每个线程打印一行问候，主线程等待全部线程结束后退出。

### 代码要点

- **参数传递**：将循环变量 `thread`（声明为 `long`）按值转换为 `void *` 传入线程，线程内再转回 `long`。传递的是值本身而非指针，因此不存在参数对象的生命周期问题。
- **句柄数组**：`thread_handles` 为每个线程保存一个句柄，供后续 `pthread_join` 使用。因线程数是运行期决定的，故用 `malloc` 在堆上分配。
- **参数解析**：用 `atoi` 将命令行参数转为整数。`atoi` 对非数字输入返回 0，后面对其进行范围检查。
- **上限保护**：每个线程都要占用栈空间与内核结构，`MAX_THREADS` 上限是基本的自我保护。

In [ ]:
%%writefile {SRC_DIR}/pthread_hello.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>

#define MAX_THREADS 64

// Shared by all threads: every thread reads the same object.
int thread_count = 0;

// Thread entry function. Its signature must be exactly void *(*)(void *).
void* Hello(void* rank) {
  long my_rank = (long)rank;
  // Output order is decided by the OS scheduler, not by creation order.
  printf("Hello from thread %ld of %d\n", my_rank, thread_count);
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <thread_count>\n", argv[0]);
    return 1;
  }

  thread_count = atoi(argv[1]);
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  // One handle per thread, needed later by pthread_join.
  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: failed to allocate thread handles\n");
    return 1;
  }

  // Fork: create the worker threads. The rank is passed by value through the
  // void * argument (valid on LP64, where a pointer is as wide as a long).
  for (long thread = 0; thread < thread_count; ++thread) {
    int rc =
        pthread_create(&thread_handles[thread], NULL, Hello, (void*)thread);
    if (rc != 0) {
      // pthread_* returns the error code directly and does not set errno.
      fprintf(stderr, "Error: pthread_create failed for thread %ld (code %d)\n",
              thread, rc);
      free(thread_handles);
      return 1;
    }
  }

  // The main thread runs concurrently with the workers.
  printf("Hello from the main thread\n");

  // Join: wait for every worker and reclaim its resources.
  for (long thread = 0; thread < thread_count; ++thread) {
    pthread_join(thread_handles[thread], NULL);
  }

  free(thread_handles);
  return 0;
}

编译并运行基础版。

In [ ]:
hello = compile_c(f"{SRC_DIR}/pthread_hello.c", f"{SRC_DIR}/pthread_hello")
print()
run_bin(hello, 4)

### 观察非确定性

下面连续运行同一个程序 20 次。**参数完全相同，且程序中不含任何随机数**，请注意每次输出的行序。

In [ ]:
from collections import Counter

orders = []
for i in range(20):
    out = run_bin(hello, 4, echo=False)
    order = tuple(
        ln.split("thread ")[1].split(" ")[0]
        for ln in out.splitlines()
        if ln.startswith("Hello from thread")
    )
    main_pos = next(k for k, ln in enumerate(out.splitlines()) if "main thread" in ln)
    orders.append(order)
    if i < 10:
        print(
            f"第 {i+1:2d} 次：工作线程输出顺序 = {list(order)}，主线程位于第 {main_pos+1} 行"
        )

tally = Counter(orders)
print(f"\n20 次运行中共出现 {len(tally)} 种不同的输出顺序：")
for o, n in tally.most_common():
    print(f"    {list(o)}  出现 {n} 次")

if len(tally) == 1:
    print(f"\n[说明] 本机核心数 = {os.cpu_count()}。")
    print("顺序每次相同并不代表顺序有保证：核心数为 1 时，线程只能被分时轮转，")
    print(
        "每个线程的工作量又极小（一次 printf），因此往往在第一个时间片内即依次执行完毕。"
    )
    print("在多核平台上重跑本单元，通常会看到多种不同的顺序。")
else:
    print("\n[说明] 同一可执行文件、同一组参数、不含任何随机数，输出顺序却不唯一。")


### 💡 结论：执行顺序不可依赖

**原因**：`pthread_create` 返回仅表示「线程已创建」，并不表示「线程已开始运行」。新线程何时被调度到核心上、运行多久被换下，完全由操作系统调度器决定。调度器的决策取决于当时的系统负载、其他进程的活动、中断到达时刻等无法控制的因素。

**这条结论贯穿全章**：

> 多线程程序的正确性，绝不能依赖于线程之间任何隐含的执行顺序。凡是需要顺序保证之处，都必须用同步机制**显式**表达出来。

在本实验中，输出乱序尚属无害现象。但是在很多情况下，同样的非确定性将直接导致**结果错误**。

## 6. 一个易被忽略的问题：线程创建中途失败

回看基础版的 Fork 阶段：

```c
for (long thread = 0; thread < thread_count; ++thread) {
  int rc = pthread_create(&thread_handles[thread], NULL, Hello, (void *)thread);
  if (rc != 0) {
    fprintf(stderr, "Error: ...");
    free(thread_handles);
    return 1;                 // 直接返回
  }
}
```

在此之前，我们默认 `pthread_create` 总会成功。但它**可能失败**：当系统线程数达到上限、内存不足、或达到 `RLIMIT_NPROC` 限制时，它会返回非 0 错误码。

### 🤔 请先自行思考，再继续阅读

> 假设要创建 10 个线程，前 5 个都成功了，第 6 个 `pthread_create` 失败。基础版此时执行 `free(thread_handles); return 1;` 直接退出。试问：**那 5 个已经创建、且正在运行的线程，将会如何？**
>
> 提示，从两个角度思考：
> 1. 这 5 个线程有没有被 `pthread_join` 回收？
> 2. `main` 返回后进程会发生什么？这 5 个线程的输出、以及它们占用的资源将如何处理？

### 6.1 问题分析

基础版在这一路径上存在一个**部分失败**（partial failure）的处理问题：

1. **已创建的线程未被汇合（join）**。失败分支直接 `return 1`，跳过了后面的 join 循环，这 5 个线程未经 `pthread_join` 便随进程一同结束。
2. **主线程 `return` 触发的是进程终止，而非线程退出**。主线程从 `main` 返回等价于调用 `exit()`，它终止的是**整个进程**。此时内核会无条件回收该进程的全部资源——地址空间、文件描述符，以及所有线程的内核栈与任务结构。

**这里需要澄清一个常见的误解**：在这种情形下，那 5 个线程**会随进程终止而被内核销毁，并不会成为「僵尸」，也不构成资源泄漏**。原因在于，「僵尸」状态的成立以**进程存活**为前提（详见 6.2）；进程既已整体终止，就不存在「记录被保留、等待回收」的状态。

因此，基础版在这一路径上的真正缺陷**不是资源泄漏**，而是：

> 主线程未对已创建的线程执行汇合便终止进程，导致这些线程被**异步地、非正常地终止**（asynchronous / abnormal termination）——它们在任意执行点被强行打断，而非运行至线程函数自然返回。

由此带来的后果属于**正确性与副作用层面**，而非资源泄漏层面：
- **工作未完成**：线程可能只执行到一半即被终止（在本实验中表现为 `printf` 输出被截断或整行丢失）。
- **副作用不完整**：若线程已修改共享状态、写入文件、或持有进程外可见的资源（如命名信号量、文件锁），这些操作可能停留在不一致的中间状态。
- **清理动作被跳过**：通过 `pthread_cleanup_push` 注册的清理处理程序、以及线程特定数据（thread-specific data）的析构函数，在强制终止下不保证被调用。

### 6.2 进程终止、线程终止与「僵尸」的严格辨析

上述分析依赖几组容易混淆的概念，本小节对它们作一次严格界定；这些概念在后续所有实验中都会用到。

#### 两种「终止」

<!--
| | 触发方式 | 影响范围 | 内核资源 |
|---|---|---|---|
| **进程终止**（process termination） | 任一线程调用 `exit()`；主线程从 `main` 返回；收到未处理的终止信号 | 终止**整个进程**及其中**全部线程** | 内核回收进程的全部资源：地址空间、文件描述符、所有线程的内核栈与任务结构 |
| **线程终止**（thread termination） | 线程调用 `pthread_exit()`；线程函数正常返回；被 `pthread_cancel` 取消 | 仅终止**该线程** | 进程继续存在；该线程资源是否立即释放，取决于它是可汇合还是分离状态 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">触发方式</th>
      <th style="text-align: left;">影响范围</th>
      <th style="text-align: left;">内核资源</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>进程终止</strong>（process termination）</td>
      <td style="text-align: left;">任一线程调用 <code>exit()</code>；主线程从 <code>main</code> 返回；收到未处理的终止信号</td>
      <td style="text-align: left;">终止<strong>整个进程</strong>及其中<strong>全部线程</strong></td>
      <td style="text-align: left;">内核回收进程的全部资源：地址空间、文件描述符、所有线程的内核栈与任务结构</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>线程终止</strong>（thread termination）</td>
      <td style="text-align: left;">线程调用 <code>pthread_exit()</code>；线程函数正常返回；被 <code>pthread_cancel</code> 取消</td>
      <td style="text-align: left;">仅终止<strong>该线程</strong></td>
      <td style="text-align: left;">进程继续存在；该线程资源是否立即释放，取决于它是可汇合还是分离状态</td>
    </tr>
  </tbody>
</table>

> 主线程从 `main` 返回（或调用 `exit()`）终止的是**整个进程**，而非仅仅主线程。POSIX 明确规定，进程内任一线程调用 `exit()` 都将导致进程连同其中所有线程一并终止。

#### 「僵尸」的精确定义及其成立前提

**僵尸（zombie）** 精确地指：一个执行体已经终止，但其**终止状态尚未被回收方读取**，因而其内核记录不能被释放，继续占据资源。该概念在进程与线程两个层面各有对应：

<!--
| | 僵尸（进程层面） | 类比（线程层面） |
|---|---|---|
| 终止的执行体 | 子进程 | 可汇合（joinable）的线程 |
| 回收动作 | 父进程调用 `wait()` / `waitpid()` | 另一线程调用 `pthread_join()` |
| 未回收时的状态 | 子进程成为**僵尸进程**（`Z` 状态），保留退出码 | 线程终止但其任务结构与栈被保留，等待被汇合 |
| 资源何时释放 | 父进程 `wait()` 之后 | 某线程 `pthread_join()` 之后 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">僵尸（进程层面）</th>
      <th style="text-align: left;">类比（线程层面）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">终止的执行体</td>
      <td style="text-align: left;">子进程</td>
      <td style="text-align: left;">可汇合（joinable）的线程</td>
    </tr>
    <tr>
      <td style="text-align: left;">回收动作</td>
      <td style="text-align: left;">父进程调用 <code>wait()</code> / <code>waitpid()</code></td>
      <td style="text-align: left;">另一线程调用 <code>pthread_join()</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">未回收时的状态</td>
      <td style="text-align: left;">子进程成为<strong>僵尸进程</strong>（<code>Z</code> 状态），保留退出码</td>
      <td style="text-align: left;">线程终止但其任务结构与栈被保留，等待被汇合</td>
    </tr>
    <tr>
      <td style="text-align: left;">资源何时释放</td>
      <td style="text-align: left;">父进程 <code>wait()</code> 之后</td>
      <td style="text-align: left;">某线程 <code>pthread_join()</code> 之后</td>
    </tr>
  </tbody>
</table>


**关键在于：僵尸状态的成立以「回收方所在的进程仍然存活」为前提。** 必须存在一个活着的进程来保留那份「待回收」的记录，僵尸才有意义。

由此可明确 6.1 的问题：当**整个进程终止**时，内核无条件回收其所有资源，包括每个线程的任务结构。此时根本不存在「记录被保留、等待回收」的状态，因而**不会产生僵尸线程，也不构成资源泄漏**。换言之，进程本身已不复存在，其内部线程的「僵尸」状态自然无从谈起。

反之，「不汇合导致泄漏」这一说法**仅在进程持续存活时才成立**：一个长期运行的进程不断创建可汇合的线程却从不 `pthread_join`，这些已终止线程的任务结构会持续累积，最终耗尽资源。

#### 分离状态（detached）：另一种正当的资源管理方式

汇合并非回收线程资源的唯一途径。若线程的返回值无人关心，可将其设为**分离状态**：

```c
pthread_t t;
pthread_create(&t, NULL, worker, arg);
pthread_detach(t);        // 或创建时通过 pthread_attr_t 设置 PTHREAD_CREATE_DETACHED
```

分离线程终止后，其资源由系统**自动回收**，无需（也不允许）再对它调用 `pthread_join`。因此避免线程资源泄漏有两条正当途径，二者择一即可：对可汇合线程调用 `pthread_join` 显式回收；或将线程设为分离状态交由系统自动回收。本章统一采用前者，因为它同时提供了同步点，便于控制并行段的边界。

#### 小结
<!--
| 场景 | 已创建线程的结局 | 是否泄漏 | 是否产生僵尸 |
|---|---|---|---|
| 创建失败后主线程 `return` / `exit()`（本实验基础版） | 随进程终止被内核销毁 | 否 | 否（进程已终止，前提不成立） |
| 进程存活，可汇合线程终止但从不 `pthread_join` | 任务结构与栈被保留 | **是** | 是（线程层面的「僵尸」） |
| 线程设为分离状态后终止 | 系统自动回收 | 否 | 否 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">场景</th>
      <th style="text-align: left;">已创建线程的结局</th>
      <th style="text-align: left;">是否泄漏</th>
      <th style="text-align: left;">是否产生僵尸</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">创建失败后主线程 <code>return</code> / <code>exit()</code>（本实验基础版）</td>
      <td style="text-align: left;">随进程终止被内核销毁</td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;">否（进程已终止，前提不成立）</td>
    </tr>
    <tr>
      <td style="text-align: left;">进程存活，可汇合线程终止但从不 <code>pthread_join</code></td>
      <td style="text-align: left;">任务结构与栈被保留</td>
      <td style="text-align: left;"><strong>是</strong></td>
      <td style="text-align: left;">是（线程层面的「僵尸」）</td>
    </tr>
    <tr>
      <td style="text-align: left;">线程设为分离状态后终止</td>
      <td style="text-align: left;">系统自动回收</td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;">否</td>
    </tr>
  </tbody>
</table>

## 7. 健壮版实现

针对 6.1 指出的问题，健壮版用一个计数器 `created` 记录**实际成功创建**的线程数，从而无论从哪条路径退出，都能对已创建的线程做对称的清理。

### 相对基础版的三处改动
<!--
| 改动 | 基础版 | 健壮版 |
|---|---|---|
| 失败时的控制流 | 直接 `return 1` | `break` 跳出循环，转入统一的清理代码 |
| join 的上界 | `thread_count` | `created`（只回收真正创建成功的线程） |
| 返回值 | 恒为 0 | `(created == thread_count) ? 0 : 1`，反映是否全部成功 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">改动</th>
      <th style="text-align: left;">基础版</th>
      <th style="text-align: left;">健壮版</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">失败时的控制流</td>
      <td style="text-align: left;">直接 <code>return 1</code></td>
      <td style="text-align: left;"><code>break</code> 跳出循环，转入统一的清理代码</td>
    </tr>
    <tr>
      <td style="text-align: left;">join 的上界</td>
      <td style="text-align: left;"><code>thread_count</code></td>
      <td style="text-align: left;"><code>created</code>（只回收真正创建成功的线程）</td>
    </tr>
    <tr>
      <td style="text-align: left;">返回值</td>
      <td style="text-align: left;">恒为 0</td>
      <td style="text-align: left;"><code>(created == thread_count) ? 0 : 1</code>，反映是否全部成功</td>
    </tr>
  </tbody>
</table>

其中第二点尤为关键：对**从未创建成功**的线程句柄（其 `pthread_t` 值未定义）调用 `pthread_join` 属未定义行为，因此上界必须是 `created` 而非 `thread_count`。

> **通用原则**：无论从哪条路径退出，都必须对称地清理已经获取的资源——已 `create` 的线程要 `join`，已 `malloc` 的内存要 `free`，已 `lock` 的锁要 `unlock`。这一原则在后续持有锁、分配缓冲区的实验中会反复出现。

In [ ]:
%%writefile {SRC_DIR}/pthread_hello_robust.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>

#define MAX_THREADS 64

// Shared by all threads: every thread reads the same object.
int thread_count = 0;

// Thread entry function. Its signature must be exactly void *(*)(void *).
void* Hello(void* rank) {
  long my_rank = (long)rank;
  // Output order is decided by the OS scheduler, not by creation order.
  printf("Hello from thread %ld of %d\n", my_rank, thread_count);
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <thread_count>\n", argv[0]);
    return 1;
  }

  thread_count = atoi(argv[1]);
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  // One handle per thread, needed later by pthread_join.
  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: failed to allocate thread handles\n");
    return 1;
  }

  // Fork: create the worker threads. Record how many actually started, so a
  // mid-loop failure still leaves every running thread accounted for.
  long created = 0;
  for (long thread = 0; thread < thread_count; ++thread) {
    int rc =
        pthread_create(&thread_handles[thread], NULL, Hello, (void*)thread);
    if (rc != 0) {
      fprintf(stderr, "Error: pthread_create failed for thread %ld (code %d)\n",
              thread, rc);
      break;  // stop creating; join the ones already created below
    }
    ++created;
  }

  // The main thread runs concurrently with the workers.
  printf("Hello from the main thread\n");

  // Join only the threads that were actually created, not thread_count.
  for (long thread = 0; thread < created; ++thread) {
    pthread_join(thread_handles[thread], NULL);
  }

  free(thread_handles);
  return (created == thread_count) ? 0 : 1;
}

编译并运行健壮版，确认正常路径下其行为与基础版一致。

In [ ]:
robust = compile_c(f"{SRC_DIR}/pthread_hello_robust.c", f"{SRC_DIR}/pthread_hello_robust")
print()
run_bin(robust, 4)

### 💡 如何真正触发一次创建失败

正常情况下创建 4 个线程不会失败。要**观察**到部分失败与清理逻辑，可用 `ulimit` 将进程可创建的线程数调低，再请求较多线程：

```bash
# 在独立子 shell 中试验，避免影响当前内核
bash -c 'ulimit -u 20; ./src_helloworld/pthread_hello_robust 64'
```

此时健壮版会打印哪个线程创建失败，并**正常 join 掉已创建的线程**后退出，返回码为 1；而基础版会跳过对已创建线程的汇合，直接退出。

> ⚠️ `ulimit -u` 限制的是「进程/线程总数」，设置过低可能导致 shell 自身也无法创建新进程。由于其效果依赖系统当时的负载，此处不写成自动执行的单元，留作练习。

## 8. 反面实验：如果不 join 会怎样

为了观察缺少 `pthread_join` 的后果，这里提供一个**反面样例** `pthread_hello_no_join.c`。它在基础版之上**刻意删去了整段 join 循环**，其余部分保持不变。

该文件在源码中明确标注了「这是反面样例、并非正确代码」，以免被误用。下面用 `%%writefile` 将其写入磁盘，随后编译运行，连续观察多次的输出行数。

In [ ]:
%%writefile {SRC_DIR}/pthread_hello_no_join.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>

#define MAX_THREADS 64

// Counterexample: this program deliberately OMITS the pthread_join loop.
// It is not correct code; it exists only to demonstrate what happens when
// the main thread returns without waiting for the workers it created.

// Shared by all threads: every thread reads the same object.
int thread_count = 0;

// Thread entry function. Its signature must be exactly void *(*)(void *).
void* Hello(void* rank) {
  long my_rank = (long)rank;
  // Output order is decided by the OS scheduler, not by creation order.
  printf("Hello from thread %ld of %d\n", my_rank, thread_count);
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <thread_count>\n", argv[0]);
    return 1;
  }

  thread_count = atoi(argv[1]);
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: failed to allocate thread handles\n");
    return 1;
  }

  // Fork: create the worker threads.
  for (long thread = 0; thread < thread_count; ++thread) {
    int rc =
        pthread_create(&thread_handles[thread], NULL, Hello, (void*)thread);
    if (rc != 0) {
      fprintf(stderr, "Error: pthread_create failed for thread %ld (code %d)\n",
              thread, rc);
      free(thread_handles);
      return 1;
    }
  }

  printf("Hello from the main thread\n");

  // NO pthread_join here. main returns immediately, terminating the whole
  // process; workers not yet scheduled are killed and their output is lost.
  free(thread_handles);
  return 0;
}

In [ ]:
no_join = compile_c(
    f"{SRC_DIR}/pthread_hello_no_join.c", f"{SRC_DIR}/pthread_hello_no_join"
)

print("\n连续运行 5 次（每次本应输出 4 行 Hello）：")
for i in range(5):
    out = run_bin(no_join, 4, echo=False)
    n = sum(1 for ln in out.splitlines() if ln.startswith("Hello from thread"))
    print(f"  第 {i+1} 次：工作线程实际输出 {n} / 4 行")


### 分析

本反面实验删去了 join 循环。主线程执行完 `return 0` 后进程立即终止，尚未被调度执行完毕的工作线程被强制终止，其输出随之丢失。因此每次运行的输出行数可能不同——这是缺少汇合所导致的**第一个后果：工作未完成**。

需要严格区分两种不同的情形，二者的后果并不相同（参见 6.2）：
- **本例（进程随即终止）**：未汇合的线程会随进程一同被内核销毁，**不会造成资源泄漏**。此处的问题是线程被异步终止、工作未完成，而非泄漏。
- **进程长期存活的情形**：若程序不退出而是持续运行（如服务端），可汇合线程终止后若始终无人 `pthread_join`，其内核任务结构与栈将一直得不到释放，形成**稳定的资源泄漏**。

综合两种情形，本章确立如下规则：

> **每一个 `pthread_create` 都应有配对的 `pthread_join`**（除非该线程被显式设置为分离状态）。

## 9. 结果分析

本实验没有性能对比，其价值在于建立三项贯穿全章的认识：

**① 执行顺序不可依赖。** 同一程序多次运行，输出顺序可能不同。多线程程序的正确性绝不能建立在任何隐含的执行顺序之上。

**② 进程终止 ≠ 线程终止，「僵尸」以进程存活为前提。** 创建失败后直接退出，已创建的线程随进程被内核销毁，既不泄漏也不产生僵尸；真正的问题是这些线程被异步终止、工作未完成。

**③ 任何退出路径都要对称清理资源。** 健壮版通过 `created` 计数与 `break`-后-统一清理，保证部分失败时已创建的线程仍被正确汇合。

### 🎓 结论

基础版与健壮版在**正常路径**上行为完全一致；差异只在**异常路径**（创建中途失败）上显现。这正是并发与系统编程的一个共性：**代码的健壮性往往不体现在正常流程，而体现在对边界与失败路径的处理**。养成「为每一条获取资源的路径准备对应的释放路径」的习惯，是写出正确并发程序的第一步。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化（建议先独立完成，再阅读思考题）：

1. 用 `bash -c 'ulimit -u 20; ./src_helloworld/pthread_hello_robust 64'` 触发一次真实的线程创建失败，观察健壮版打印哪个线程失败、以及它如何 join 掉已创建的线程；再用基础版 `pthread_hello` 重复，对比两者行为的差异。
2. 将线程入口函数的签名改为 `void Hello(long rank)`，尝试编译，记录编译器的错误信息，解释为什么签名不能更改。
3. 将 `(void *)thread` 改为 `(void *)&thread`，运行多次，记录输出中 `my_rank` 的取值，解释为什么会出现这种结果。
4. 基础版用 `atoi` 解析参数，输入 `5x` 会被解析为 5 而不报错。请将其改用 `strtol` 并检查 `endptr`，使 `5x` 这类输入被拒绝。
5. 将线程数依次设为 1、2、4、8、16、32，各运行 5 次，统计输出顺序与创建顺序完全一致的比例，解释该比例随线程数变化的趋势。

## 11. 🤔 思考题

- `pthread_create` 返回 0 之后，新线程是否一定已经开始执行？若不是，程序为何仍然正确？
- 健壮版在创建失败时用 `break` 而非 `return`。若改回 `return 1`，会退回到基础版的哪个问题？
- 本实验中多个线程同时调用 `printf` 写同一标准输出，为何没有出现半行字符交错的情况？（提示：查阅 C 标准库对 `FILE` 对象的线程安全规定。）
- 线程共享全局区与堆、私有栈。那么一个线程能否读写另一线程栈上的局部变量？若能，需要什么条件？这样做是否安全？
- 健壮版对「已创建的线程要 join、已分配的内存要 free」做了对称清理。若 `Hello` 线程内部还申请了一块内存，这块内存应由谁、在何时释放？请说明理由。

## 12. 小结与后续

本实验完成了「**基础实现 → 提出问题 → 健壮实现**」的完整过程：
<!--
| 版本 | 新增内容 | 涉及知识点 |
|---|---|---|
| **基础版** | `Hello` + Fork-Join 框架 | 线程创建/汇合、参数传递、非确定性 |
| **健壮版** | `created` 计数 + 对称清理 | 部分失败处理、进程/线程终止、僵尸与分离状态 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>基础版</strong></td>
      <td style="text-align: left;"><code>Hello</code> + Fork-Join 框架</td>
      <td style="text-align: left;">线程创建/汇合、参数传递、非确定性</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>健壮版</strong></td>
      <td style="text-align: left;"><code>created</code> 计数 + 对称清理</td>
      <td style="text-align: left;">部分失败处理、进程/线程终止、僵尸与分离状态</td>
    </tr>
  </tbody>
</table>

通过 Hello World，我们建立了 Pthreads 编程的最小框架，并认识到两个贯穿全章的事实：**执行顺序不可依赖**，以及**任何退出路径都要对称清理资源**。

➡️ **后续内容：实验二 矩阵向量乘法与线程传参**。当线程需要接收多个参数时，将遇到第一个实质性问题：**如何安全地向线程传递数据**——这直接关系到本实验中「按值传递 vs 传地址」的抉择。